# Image captioning workflow

Use the refactored package for data preparation, training, evaluation, and inference. See the repository README for installation and data setup. This notebook does not include trained weights or experiment outputs.


In [ ]:
from pathlib import Path

# Open from the repository root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / "configs/default.json").exists() and ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/default.json").exists():
    raise RuntimeError("Start Jupyter inside the image-captioning repository")

from image_captioning.config import Config
from image_captioning.data import read_annotations, split_records
from image_captioning.vocabulary import Vocabulary

config = Config.load(ROOT / "configs/default.json")
# Resolve local paths so running from notebooks/ is also supported.
for name in ("captions_file", "image_dir", "output_dir"):
    path = Path(getattr(config, name))
    if not path.is_absolute():
        setattr(config, name, str(ROOT / path))
config


## Inspect the data and training vocabulary

Point the configuration to your Flickr30k CSV and image folder before running. All captions for one image remain in the same split.


In [ ]:
records = read_annotations(config.captions_file)
splits = split_records(records, config.validation_fraction, config.test_fraction, config.seed)
train_captions = [text for record in splits["train"] for text in record.captions]
vocabulary = Vocabulary.build(train_captions, config.min_frequency)
print({name: len(items) for name, items in splits.items()})
print("Vocabulary size:", len(vocabulary))
print("Example:", train_captions[0])


## Train (explicit opt-in)

Training uses your local data and may download ImageNet encoder weights. The output directory must be empty. Change the flag only when ready to run an experiment. On platforms with multiprocessing issues, set `config.num_workers = 0`.


In [ ]:
from image_captioning.train import train

RUN_TRAINING = False
if RUN_TRAINING:
    history = train(config)
else:
    print("Training is disabled. Set RUN_TRAINING = True to start.")


## Plot a completed run

Install the optional notebook dependencies first. These figures read history produced by this package, not historical Kaggle output.


In [ ]:
from image_captioning.visualization import plot_history

history_path = Path(config.output_dir) / "history.json"
if history_path.exists():
    plot_history(history_path)
else:
    print("Train a model first to create history.json.")


## Evaluate a completed run

This command uses the checkpoint's saved test image IDs and the original annotations. Loss and perplexity are teacher-forced, padding-aware token metrics.


In [ ]:
import subprocess
import sys

checkpoint = Path(config.output_dir) / "best.pt"
if checkpoint.exists():
    subprocess.run([sys.executable, "-m", "image_captioning.evaluate",
                    "--checkpoint", str(checkpoint),
                    "--captions-file", config.captions_file,
                    "--image-dir", config.image_dir, "--num-workers", "0"], check=True)
else:
    print("Train a model first to create best.pt.")


## Caption a new image

Generation starts from `<SOS>` and feeds back generated tokens without a reference caption. Choose your own image after training.


In [ ]:
from PIL import Image

from image_captioning.checkpoint import load_model
from image_captioning.data import make_transforms
from image_captioning.predict import generate_caption
from image_captioning.utils import select_device
from image_captioning.visualization import show_image

IMAGE_PATH = None  # e.g. ROOT / "data/my_photo.jpg"
if IMAGE_PATH is not None and checkpoint.exists():
    model, vocab, trained_config, _ = load_model(checkpoint, select_device())
    with Image.open(IMAGE_PATH) as image:
        tensor = make_transforms(trained_config.resize, trained_config.crop_size)(image.convert("RGB"))
    caption = generate_caption(model, tensor, vocab, min(20, trained_config.max_length), trained_config.max_length)
    show_image(tensor, caption)
else:
    print("Set IMAGE_PATH and train a model first.")
